### Réplication du Fedlocker (https://jnathan9.github.io/fedlock/) en version amateur mini sur base données https://cbspeeches.com/

In [ ]:
import pandas as pd
from pathlib import Path

# Use the current project folder for portable file paths
base_dir = Path.cwd()
source_file = base_dir / "CBS_dataset_v1.0.csv"
lite_file = base_dir / "ecb_speeches_lite.csv"

if not source_file.exists():
    raise FileNotFoundError(f"Source file not found: {source_file}")

# Chargement intelligent
df = pd.read_csv(source_file)

# 1. On garde uniquement la BCE
df_ecb = df[df['CentralBank'] == 'European Central Bank'].copy()

# 2. On cible les speakers majeurs
# Tu peux ajouter d'autres noms selon le besoin
top_speakers = ['Christine Lagarde', 'Mario Draghi', 'Isabel Schnabel', 'Philip R. Lane']
df_filtered = df_ecb[df_ecb['Authorname'].isin(top_speakers)].copy()

# 3. Nettoyage date
df_filtered['Date'] = pd.to_datetime(df_filtered['Date'])

# Sauvegarde d'un dataset de travail léger (quelques Mo au lieu de 650)
df_filtered.to_csv(lite_file, index=False)
print(f"Fichier prêt : {len(df_filtered)} discours conservés.")
print(f"Chemin de sortie : {lite_file}")

In [ ]:
import pandas as pd
from pathlib import Path

lite_file = Path.cwd() / "ecb_speeches_lite.csv"
if not lite_file.exists():
    raise FileNotFoundError(f"Lite dataset not found: {lite_file}. Run cell 2 first.")

df = pd.read_csv(lite_file)

# Vue d'ensemble
print(df['Authorname'].value_counts())
print()
print(df.groupby(['Authorname', df['Date'].str[:4]]).size().unstack(fill_value=0))
print()
print(f"Période : {df['Date'].min()} → {df['Date'].max()}")
print(f"Longueur moyenne des textes : {df['text'].str.len().mean():.0f} caractères")
print(f"Textes manquants : {df['text'].isna().sum()}")

check main stat. on speeches length

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

lite_file = Path.cwd() / "ecb_speeches_lite.csv"
if not lite_file.exists():
    raise FileNotFoundError(f"Lite dataset not found: {lite_file}. Run cell 2 first.")

# Load your lite dataset
df = pd.read_csv(lite_file)

# Calculate word counts
df['word_count'] = df['text'].str.split().str.len()

# Calculate statistics
stats = {
    "Mean": df['word_count'].mean(),
    "Median": df['word_count'].median(),
    "75th Percentile": df['word_count'].quantile(0.75),
    "95th Percentile": df['word_count'].quantile(0.95),
    "Max": df['word_count'].max()
}

print("--- Speech Length Statistics (Words) ---")
for key, val in stats.items():
    print(f"{key:15}: {val:.0f} words")

# Visualize the distribution
plt.figure(figsize=(10, 6))
plt.hist(df['word_count'], bins=30, color='skyblue', edgecolor='black')
plt.title('Distribution of ECB Speech Word Counts')
plt.xlabel('Number of Words')
plt.ylabel('Number of Speeches')
plt.axvline(stats['Median'], color='red', linestyle='dashed', label=f"Median: {stats['Median']:.0f}")
plt.legend()
plt.show()

print("\nRecommendation:")
if stats['95th Percentile'] <= 5000:
    print("-> 5,000 words is perfect. It covers almost every speech entirely.")
else:
    print(f"-> Consider increasing to {int(stats['95th Percentile'])} words to capture the full context of longer speeches.")

In [ ]:
%pip install -U google-generativeai

Version avec api key gemini

In [ ]:
import os
print("GEMINI_API_KEY loaded:", bool(os.getenv("GEMINI_API_KEY")))


In [ ]:
import json
import os
import re
import time
from pathlib import Path

import google.generativeai as genai
import pandas as pd

# -- 1. Configuration ----------------------------------------------------------
BASE_DIR = Path.cwd()
INPUT_FILE = BASE_DIR / "ecb_speeches_lite.csv"
OUTPUT_FILE = BASE_DIR / "ecb_speeches_scored.csv"

API_KEY = os.getenv("GEMINI_API_KEY", "").strip()
model = None
if API_KEY:
    genai.configure(api_key=API_KEY)
    model = genai.GenerativeModel("gemini-2.0-flash-lite")
else:
    print("GEMINI_API_KEY is not set. The run will stop safely after saving progress.")

MAX_WORDS = 5000
SAVE_EVERY = 10
REQUEST_PAUSE_SECONDS = 2
MAX_RETRIES = 3
BASE_BACKOFF_SECONDS = 4

# -- 2. Macro Context ----------------------------------------------------------
ecb_rates = {
    2011: 1.25,
    2012: 0.75,
    2013: 0.25,
    2014: 0.05,
    2015: 0.05,
    2016: 0.00,
    2017: 0.00,
    2018: 0.00,
    2019: 0.00,
    2020: 0.00,
    2021: 0.00,
    2022: 1.50,
    2023: 4.00,
}


def _is_quota_error(error_text: str) -> bool:
    lowered = error_text.lower()
    return (
        "quota" in lowered
        or "429" in lowered
        or "resource_exhausted" in lowered
        or "rate limit" in lowered
    )


def _retry_after_seconds(error_text: str, default_seconds: int = 60) -> int:
    match = re.search(r"retry in\s+([0-9]+(?:\.[0-9]+)?)s", error_text.lower())
    if not match:
        return default_seconds
    return max(1, int(float(match.group(1)) + 1))


def get_sentiment(text, speaker, date):
    if model is None:
        return {
            "score": None,
            "reason": "Missing GEMINI_API_KEY. Set it and rerun this cell.",
            "fatal_quota": True,
            "retry_after_seconds": 0,
        }

    words = str(text).split()
    context_text = " ".join(words[:MAX_WORDS])

    year = pd.to_datetime(date).year
    rate = ecb_rates.get(year, 0.0)

    prompt = f"""You are an expert in ECB monetary policy communication.
Analyze the speech for hawk-dove orientation on a scale from -1.0 to +1.0.

- Hawk (+1.0): inflation control, tightening, concern about high prices.
- Dove (-1.0): growth/employment support, easing, concern about low inflation.

Context calibration:
If the speaker urges caution on rate cuts when inflation is low, that is hawkish.
The same language when inflation is high is neutral.

Speaker: {speaker}
Date: {date}
ECB Main Rate at time: {rate}%

Speech Text:
{context_text}"""

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = model.generate_content(
                prompt,
                generation_config=genai.GenerationConfig(
                    response_mime_type="application/json",
                    response_schema={
                        "type": "object",
                        "properties": {
                            "score": {"type": "number"},
                            "reason": {"type": "string"},
                        },
                        "required": ["score", "reason"],
                    },
                ),
            )
            payload = json.loads(response.text)

            score = payload.get("score")
            if score is not None:
                score = float(score)
                score = max(-1.0, min(1.0, score))

            return {
                "score": score,
                "reason": str(payload.get("reason", "")).strip(),
                "fatal_quota": False,
            }

        except Exception as e:
            error_text = str(e)

            if _is_quota_error(error_text):
                return {
                    "score": None,
                    "reason": error_text,
                    "fatal_quota": True,
                    "retry_after_seconds": _retry_after_seconds(error_text),
                }

            if attempt == MAX_RETRIES:
                return {"score": None, "reason": error_text, "fatal_quota": False}

            sleep_s = BASE_BACKOFF_SECONDS * attempt
            print(f"\n  Temporary API error. Retry {attempt}/{MAX_RETRIES} in {sleep_s}s")
            time.sleep(sleep_s)


# -- 3. Load Data and Resume Logic --------------------------------------------
if not INPUT_FILE.exists():
    raise FileNotFoundError(f"Input file not found: {INPUT_FILE}")

if OUTPUT_FILE.exists():
    df_scored = pd.read_csv(OUTPUT_FILE)
    print(f"Resuming from: {OUTPUT_FILE}")
else:
    df_scored = pd.read_csv(INPUT_FILE)
    print(f"Starting fresh from: {INPUT_FILE}")

if "hawk_dove_score" not in df_scored.columns:
    df_scored["hawk_dove_score"] = None
if "analysis_reason" not in df_scored.columns:
    df_scored["analysis_reason"] = None

df_scored["Date"] = pd.to_datetime(df_scored["Date"], errors="coerce")
processed = int(df_scored["hawk_dove_score"].notna().sum())
print(f"Already processed: {processed}")

# -- 4. Main Loop --------------------------------------------------------------
to_score = df_scored[df_scored["hawk_dove_score"].isna()].index
print(f"Total speeches to process: {len(to_score)}\n")

stopped_on_quota = False

for i, idx in enumerate(to_score, start=1):
    row = df_scored.loc[idx]
    print(
        f"[{i}/{len(to_score)}] {row['Authorname']} ({row['Date'].strftime('%Y-%m')}) ",
        end="",
        flush=True,
    )

    result = get_sentiment(row["text"], row["Authorname"], row["Date"])

    df_scored.at[idx, "hawk_dove_score"] = result.get("score")
    df_scored.at[idx, "analysis_reason"] = result.get("reason")
    print(f"-> Score: {result.get('score')}")

    # Save periodically to avoid losing progress.
    if i % SAVE_EVERY == 0:
        df_scored.to_csv(OUTPUT_FILE, index=False)
        print("  Progress saved.")

    if result.get("fatal_quota"):
        wait_s = int(result.get("retry_after_seconds", 0))
        df_scored.to_csv(OUTPUT_FILE, index=False)
        print("  Quota or key issue detected. Progress saved and run stopped.")
        if wait_s > 0:
            print(f"  Retry in about {wait_s}s, enable billing, or switch to the Groq cell.")
        else:
            print("  Set GEMINI_API_KEY and rerun this cell.")
        stopped_on_quota = True
        break

    time.sleep(REQUEST_PAUSE_SECONDS)

# Final save
df_scored.to_csv(OUTPUT_FILE, index=False)

if stopped_on_quota:
    print(f"\nStopped early. Partial results are in: {OUTPUT_FILE}")
else:
    print(f"\nDone. Results saved to: {OUTPUT_FILE}")

In [ ]:
%pip install Groq

Version avec api key groq

In [ ]:
import pandas as pd
from groq import Groq
import json
import time
import os

# ── 1. Configuration ──────────────────────────────────────────────────────────
client = Groq(api_key=os.getenv("GROQ_API_KEY", ""))

df = pd.read_csv('ecb_speeches_lite.csv')
df['Date'] = pd.to_datetime(df['Date'])

# ── 2. Contexte macro : taux directeur BCE par année ─────────────────────────
ecb_rates = {
    2011: 1.25, 2012: 0.75, 2013: 0.25, 2014: 0.05,
    2015: 0.05, 2016: 0.00, 2017: 0.00, 2018: 0.00,
    2019: 0.00, 2020: 0.00, 2021: 0.00, 2022: 1.50,
    2023: 4.00
}

# ── 3. Fonction de scoring ────────────────────────────────────────────────────
def get_sentiment(text, speaker, date):
    context = text[:5000]
    year = pd.to_datetime(date).year
    rate = ecb_rates.get(year, 0.0)

    prompt = f"""You are an expert in ECB monetary policy communication.

Score this speech on a hawkish-dovish scale from -1.0 to +1.0 (0 = neutral).
Calibrate relative to context: urging caution on cuts when inflation is low = hawkish.
The same language when inflation is high = neutral or dovish.

Speaker: {speaker}
Date: {date}
ECB main rate at the time: {rate}%

Speech:
{context}

Respond ONLY with valid JSON:
{{"score": 0.5, "reason": "Short explanation in English"}}"""

    try:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}],
            response_format={"type": "json_object"},  # JSON garanti, pas besoin de regex
            temperature=0.1  # On veut du déterministe, pas de créativité
        )
        result = json.loads(response.choices[0].message.content)
        score = result.get('score')
        if score is not None and -1.0 <= float(score) <= 1.0:
            return result
        else:
            print(f"  ⚠ Score hors bornes : {score}")
            return {"score": None, "reason": f"Score invalide : {score}"}

    except Exception as e:
        print(f"  ⚠ Erreur API : {e}")
        return {"score": None, "reason": str(e)}

# ── 4. Reprise automatique si interruption ────────────────────────────────────
try:
    df_scored = pd.read_csv('ecb_speeches_scored.csv')
    df_scored['Date'] = pd.to_datetime(df_scored['Date'])
    already_done = df_scored['hawk_dove_score'].notna().sum()
    print(f"Reprise détectée : {already_done} discours déjà scorés.")
except FileNotFoundError:
    df_scored = df.copy()
    df_scored['hawk_dove_score'] = None
    df_scored['analysis_reason'] = None
    print("Nouveau run.")

# ── 5. Boucle de scoring ──────────────────────────────────────────────────────
to_score = df_scored[df_scored['hawk_dove_score'].isna()].index

# ⚠ Test sur 10 d'abord — commente cette ligne pour le run complet
# to_score = to_score[:10]

print(f"Discours à scorer : {len(to_score)}\n")

for i, idx in enumerate(to_score):
    row = df_scored.loc[idx]
    print(f"[{i+1}/{len(to_score)}] {row['Authorname']} — {row['Date'].strftime('%Y-%m')} ", end="", flush=True)

    result = get_sentiment(row['text'], row['Authorname'], row['Date'])

    df_scored.at[idx, 'hawk_dove_score'] = result['score']
    df_scored.at[idx, 'analysis_reason'] = result['reason']

    print(f"→ {result['score']}")

    if (i + 1) % 10 == 0:
        df_scored.to_csv('ecb_speeches_scored.csv', index=False)
        print(f"  💾 Sauvegarde ({i+1} traités)\n")

    time.sleep(0.3)  # Groq est rapide, 0.3s suffit

df_scored.to_csv('ecb_speeches_scored.csv', index=False)

# ── 6. Rapport final ──────────────────────────────────────────────────────────
scored = df_scored['hawk_dove_score'].notna().sum()
errors = df_scored['hawk_dove_score'].isna().sum()
print(f"\n✅ Terminé : {scored} scorés, {errors} échecs")
if errors > 0:
    print("Discours en échec :")
    print(df_scored[df_scored['hawk_dove_score'].isna()][['Authorname', 'Date']])